In [1]:
import math
import time

N = 2_000_000
datos = [i / 100 + 1 for i in range(N)]

print(f"Cantidad de datos: {len(datos)}")
print(f"Primer dato: {datos[0]}")
print(f"Último dato: {datos[-1]}")


Cantidad de datos: 2000000
Primer dato: 1.0
Último dato: 20000.99


In [2]:
%%writefile paralelo_utils.py
import math
import time
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor


def funcion(x):
    return math.sqrt(x) + x**2 + math.sin(x) + math.cos(x) + math.log(x)


def procesar_bloque(bloque):
    return [funcion(x) for x in bloque]


def ejecutar_paralelo(datos, workers):
    if workers < 1:
        raise ValueError("Debe haber al menos un worker.")

    inicio = time.perf_counter()

    # Repartir los datos en bloques conservando el orden.
    bloques = [
        datos[i * len(datos) // workers:
              (i + 1) * len(datos) // workers]
        for i in range(workers)
    ]

    contexto = mp.get_context("spawn")

    with ProcessPoolExecutor(
        max_workers=workers,
        mp_context=contexto
    ) as executor:
        resultados = [
            valor
            for bloque in executor.map(procesar_bloque, bloques)
            for valor in bloque
        ]

    tiempo = time.perf_counter() - inicio
    return resultados, tiempo

Writing paralelo_utils.py


In [3]:
from paralelo_utils import ejecutar_paralelo, funcion

tiempos = {}

for workers in (1, 2, 4):
    print(f"Procesando con {workers} worker(s)...", flush=True)

    resultados, tiempo = ejecutar_paralelo(datos, workers)
    tiempos[workers] = tiempo

    print(f"Tiempo: {tiempo:.6f} segundos")
    print(f"Cantidad de resultados: {len(resultados)}")
    print(f"Primer resultado: {resultados[0]}")
    print(f"Último resultado: {resultados[-1]}")
    print()

    del resultados

Procesando con 1 worker(s)...
Tiempo: 2.205312 segundos
Cantidad de resultados: 2000000
Primer resultado: 3.3817732906760365
Último resultado: 400039753.26731867

Procesando con 2 worker(s)...
Tiempo: 1.165061 segundos
Cantidad de resultados: 2000000
Primer resultado: 3.3817732906760365
Último resultado: 400039753.26731867

Procesando con 4 worker(s)...
Tiempo: 0.935477 segundos
Cantidad de resultados: 2000000
Primer resultado: 3.3817732906760365
Último resultado: 400039753.26731867



In [4]:
# Validación fuera de las mediciones de rendimiento.
referencia = [funcion(x) for x in datos]

for workers in (1, 2, 4):
    resultados, _ = ejecutar_paralelo(datos, workers)

    assert resultados == referencia, (
        f"Los resultados no coinciden con {workers} workers"
    )

    print(f"{workers} worker(s): los 2,000,000 resultados coinciden.")
    del resultados

del referencia

1 worker(s): los 2,000,000 resultados coinciden.
2 worker(s): los 2,000,000 resultados coinciden.
4 worker(s): los 2,000,000 resultados coinciden.


In [5]:
# Comprobar que el reparto no pierde datos y conserva el orden.
for cantidad in (1, 7, 101):
    datos_prueba = [i / 100 + 1 for i in range(cantidad)]
    referencia_prueba = [funcion(x) for x in datos_prueba]

    for workers in (1, 2, 4):
        resultado_prueba, _ = ejecutar_paralelo(datos_prueba, workers)

        assert resultado_prueba == referencia_prueba, (
            f"Error con {cantidad} datos y {workers} workers"
        )

    print(f"Validación correcta con {cantidad} datos.")

del datos_prueba, referencia_prueba, resultado_prueba

Validación correcta con 1 datos.
Validación correcta con 7 datos.
Validación correcta con 101 datos.
